## Preparation of Tile for the Inference Phase

In [7]:
import pandas as pd
import os
from osgeo import gdal
from joblib import dump
import numpy as np
import matplotlib.pyplot as plt
import time


In [8]:
def arrays_to_raster(arrays, cols, rows, transform, projection, output_name):
    """Converts list of NumPy arrays to a multi-band raster file, overwriting existing file if it exists."""
    bands = len(arrays)
    driver = gdal.GetDriverByName('GTiff')

    # Check if the output file already exists, and delete it if it does
    if os.path.exists(output_name):
        os.remove(output_name)

    dataset_index = driver.Create(output_name, cols, rows, bands, gdal.GDT_Float32)
    dataset_index.SetGeoTransform(transform)
    dataset_index.SetProjection(projection)
    
    for i, array in enumerate(arrays, start=1):
        band = dataset_index.GetRasterBand(i)
        band.WriteArray(array)
    dataset_index.FlushCache()

    return dataset_index

### Calculate NDVI and NBR indices

In [9]:
def calculate_indices(dsimage):
    """Calculates various spectral indices from raster bands."""
    array1 = dsimage.GetRasterBand(1).ReadAsArray().astype(float)  # Blue
    array2 = dsimage.GetRasterBand(2).ReadAsArray().astype(float)  # Green
    array3 = dsimage.GetRasterBand(3).ReadAsArray().astype(float)  # Red
    array4 = dsimage.GetRasterBand(4).ReadAsArray().astype(float)  # NIR
    array5 = dsimage.GetRasterBand(5).ReadAsArray().astype(float)  # SWIR1
    array6 = dsimage.GetRasterBand(6).ReadAsArray().astype(float)  # SWIR2

    # Avoid division by zero
    ndvi = np.where((array4 + array3) == 0, 0, (array4 - array3) / (array4 + array3))
    nbr = np.where((array4 + array6) == 0, 0, (array4 - array6) / (array4 + array6))
    tcb = 0.3029 * array1 + 0.2786 * array2 + 0.4733 * array3 + 0.5599 * array4 + 0.508 * array5 + 0.1872 * array6
    tcg = -0.2941 * array1 - 0.243 * array2 - 0.5424 * array3 + 0.7276 * array4 + 0.0713 * array5 - 0.1608 * array6
    tcw = 0.1511 * array1 + 0.1973 * array2 + 0.3283 * array3 + 0.3407 * array4 - 0.7117 * array5 - 0.4559 * array6
    tc_di = tcb - (tcg + tcw) # Disturbance index: (Healey et al., 2005)
    
    # Normalized indices
    # mean und std von allen Dateien (!)
    tcb_n = (tcb - np.mean(tcb)) / np.std(tcb)
    tcg_n = (tcg - np.mean(tcg)) / np.std(tcg)
    tcw_n = (tcw - np.mean(tcw)) / np.std(tcw)
    tc_di_n = tcb_n - (tcg_n + tcw_n)    

    return {
        "NBR": nbr,
        "NDVI": ndvi,
        "TCB": tcb,
        "TCG": tcg,
        "TCW": tcw,
        #"TC_DI": tc_di,
        "TC_DI_N": tc_di_n
    }

In [10]:
def process_images_in_folder(worksp):
    """ Process images, only stacking NDVI & NBR indices per year. """
    gdal.AllRegister()
    start_time = time.time()
    
    # Gather and sort file names
    files = sorted(f for f in os.listdir(worksp) if f.endswith('801_LEVEL3_LNDLG_BAP.tif'))
    workspace_saving = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI'

    if not os.path.exists(workspace_saving):
        os.makedirs(workspace_saving)
    else:
        # Clear existing files
        for file in os.listdir(workspace_saving):
            file_path = os.path.join(workspace_saving, file)
            if os.path.isfile(file_path):
                os.remove(file_path)
    
    # Organize files by year
    year_groups = {}
    for file in files:
        year = file[:4]  # Extract first 4 characters as the year
        if year not in year_groups:
            year_groups[year] = []
        year_groups[year].append(file)
    
    # Process each year group
    for year, file_group in year_groups.items():
        all_stacked_data = []
        first_image = True

        for f in file_group:
            image_path = os.path.join(worksp, f)
            print(f"Processing image: {image_path}")
            dsimage = gdal.Open(image_path, gdal.GA_ReadOnly)
            if dsimage is None:
                print(f"Failed to open image: {image_path}")
                continue

            if first_image:
                cols, rows = dsimage.RasterXSize, dsimage.RasterYSize
                transform = dsimage.GetGeoTransform()
                projection = dsimage.GetProjection()
                first_image = False

            # **Only Calculate Indices (No Extra Bands)**
            indices = calculate_indices(dsimage)
            stacked_bands = np.stack([
                        indices["NDVI"], 
                        indices["NBR"],
                        indices["TCB"],
                        indices["TCG"],
                        indices["TCW"],
                        indices["TC_DI_N"]
                        ], axis=0)  # Now 6 bands

            all_stacked_data.append(stacked_bands)

        # Convert list to NumPy array
        all_stacked_data = np.array(all_stacked_data)

        # If multiple images per year, average them
        final_stacked = np.mean(all_stacked_data, axis=0)

        # Save the raster
        output_filename = f"{year}_stacked_NDVI_NBR.tif"
        output_path = os.path.join(workspace_saving, output_filename)
        
        arrays_to_raster(final_stacked, cols, rows, transform, projection, output_path)
        print(f"Saved: {output_path}")

    print(f"Process completed in {time.time() - start_time} seconds")


In [11]:
workspace = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data'
process_images_in_folder(workspace)

Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19840801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1984_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19850801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1985_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19860801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1986_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19870801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1987_stacked_NDVI_NBR.tif
Processing image: /home/ubun

/tmp/ipykernel_429237/2654257394.py:12: RuntimeWarning: divide by zero encountered in divide
  nbr = np.where((array4 + array6) == 0, 0, (array4 - array6) / (array4 + array6))


Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1990_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19910801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1991_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19920801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1992_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19930801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/1993_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/19940801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/sav

/tmp/ipykernel_429237/2654257394.py:12: RuntimeWarning: invalid value encountered in divide
  nbr = np.where((array4 + array6) == 0, 0, (array4 - array6) / (array4 + array6))


Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2007_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20080801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2008_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20090801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2009_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20100801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2010_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20110801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/sav

/tmp/ipykernel_429237/2654257394.py:11: RuntimeWarning: divide by zero encountered in divide
  ndvi = np.where((array4 + array3) == 0, 0, (array4 - array3) / (array4 + array3))


Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2013_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20140801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2014_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20150801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2015_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20160801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2016_stacked_NDVI_NBR.tif
Processing image: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/raw_data/20170801_LEVEL3_LNDLG_BAP.tif
Saved: /home/ubuntu/work/sav

In [12]:
import functools
import rasterio
import os
import numpy

def compare(f1, f2):
    f1 = int(f1.split('_')[0])
    f2 = int(f2.split('_')[0])
    if f1 < f2:
           return -1
    elif f1 == f2:
           return 0
    else:
           return 1
        
def stack_all():
        path = r'../tiles/NDVI/'
        #print(os.listdir(path))
        files = [f for f in os.listdir(path)]
        files = sorted(files, key=functools.cmp_to_key(compare))

        all_data = []
        for f in files:
                if f.endswith('.tif'):
                        fpath = os.path.join(path, f)
                        print(fpath)
                        with rasterio.open(fpath) as src1:
                                data = src1.read()
                                all_data.append(data)
                                                 
        all_data_array = numpy.stack(all_data, axis=0)
        print(all_data_array.shape)
        return all_data_array
        #for file in workspace:


        #print(data.shape)
        
ndvi_nbr_arr = stack_all()

../tiles/NDVI/1984_stacked_NDVI_NBR.tif


../tiles/NDVI/1985_stacked_NDVI_NBR.tif
../tiles/NDVI/1986_stacked_NDVI_NBR.tif
../tiles/NDVI/1987_stacked_NDVI_NBR.tif
../tiles/NDVI/1988_stacked_NDVI_NBR.tif
../tiles/NDVI/1989_stacked_NDVI_NBR.tif
../tiles/NDVI/1990_stacked_NDVI_NBR.tif
../tiles/NDVI/1991_stacked_NDVI_NBR.tif
../tiles/NDVI/1992_stacked_NDVI_NBR.tif
../tiles/NDVI/1993_stacked_NDVI_NBR.tif
../tiles/NDVI/1994_stacked_NDVI_NBR.tif
../tiles/NDVI/1995_stacked_NDVI_NBR.tif
../tiles/NDVI/1996_stacked_NDVI_NBR.tif
../tiles/NDVI/1997_stacked_NDVI_NBR.tif
../tiles/NDVI/1998_stacked_NDVI_NBR.tif
../tiles/NDVI/1999_stacked_NDVI_NBR.tif
../tiles/NDVI/2000_stacked_NDVI_NBR.tif
../tiles/NDVI/2001_stacked_NDVI_NBR.tif
../tiles/NDVI/2002_stacked_NDVI_NBR.tif
../tiles/NDVI/2003_stacked_NDVI_NBR.tif
../tiles/NDVI/2004_stacked_NDVI_NBR.tif
../tiles/NDVI/2005_stacked_NDVI_NBR.tif
../tiles/NDVI/2006_stacked_NDVI_NBR.tif
../tiles/NDVI/2007_stacked_NDVI_NBR.tif
../tiles/NDVI/2008_stacked_NDVI_NBR.tif
../tiles/NDVI/2009_stacked_NDVI_NBR.tif


In [13]:
def process_tile(ndvi_arr):
    print(ndvi_arr.shape)
    diff_arr = np.zeros((ndvi_arr.shape[0]-1, ndvi_arr.shape[1], ndvi_arr.shape[2], ndvi_arr.shape[3]))
    print(diff_arr.shape)
    # Calculate the differences between the current row and the previous row
    diff_arr = ndvi_arr[1:,:,:,:] - ndvi_arr[:-1,:,:,:]
    #for i in range(1, ndvi_arr.shape[0]):
    #    diff_arr[i-1,:,:,:] = ndvi_arr[i,:,:,:] - ndvi_arr[i-1,:,:,:]
    return diff_arr

diff_arr = process_tile(ndvi_nbr_arr)

(40, 6, 5000, 5000)
(39, 6, 5000, 5000)


### Stack differences and values together

In [14]:
# remove values of first year
ndvi_nbr_arr = ndvi_nbr_arr[1:,:,:,:]

#stack ndvi values and differences
ndvi_diff_arr = np.concatenate((ndvi_nbr_arr, diff_arr), axis=1)

In [15]:
print(ndvi_diff_arr.shape)

(39, 12, 5000, 5000)


In [16]:
# Save the ndvi_diff_arr array
#save_path_diff = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/ndvi_diff_arr.npz'
#np.savez_compressed(save_path_diff, data=ndvi_diff_arr)
#print(f"Saved ndvi_diff_arr to: {save_path_diff}")

Saved ndvi_diff_arr to: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/ndvi_diff_arr.npz


: 

In [ ]:
print(ndvi_diff_arr.shape)

# Reshape to have all years for each pixel stacked vertically
reshaped_images = ndvi_diff_arr.transpose(2, 3, 0, 1).reshape(-1, 4)
#reshaped_images = ndvi_diff_arr.transpose(2, 3, 0, 1).reshape(-1, 4)

reshaped_images_time = ndvi_diff_arr.transpose(0, 2, 3, 1).reshape(-1, 4)

print(f"Reshaped array shape: {reshaped_images.shape}")
print(f"Reshaped array shape: {reshaped_images_time.shape}")

(39, 12, 5000, 5000)


In [13]:
# Save the reshaped array
save_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/reshaped_tile_for_inference.npz'
np.savez_compressed(save_path, data=reshaped_images)
print(f"Saved reshaped array to: {save_path}")

# Save the reshaped array
save_path_time = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/reshaped_tile_for_inference_time.npz'
np.savez_compressed(save_path_time, data=reshaped_images_time)
print(f"Saved reshaped array to: {save_path_time}")

Saved reshaped array to: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/reshaped_tile_for_inference.npz


In [12]:
print(reshaped_images.shape)

# Verification of reshaping
def verify_reshaping():
    # Check data for a specific pixel location (e.g., x=100, y=100)
    x, y = 100, 100
    
    # Get original data for this pixel across all years
    original_pixel_data = ndvi_diff_arr[:, :, x, y]  # Shape should be (39, 4)
    print("\nOriginal data shape for pixel (100,100):", original_pixel_data.shape)
    
    # Get reshaped data for the same pixel
    pixel_index = (x * 5000 + y) * 39  # Calculate starting index for this pixel
    reshaped_pixel_data = reshaped_images[pixel_index:pixel_index + 39]  # Get all 39 years for this pixel
    print("Reshaped data shape for pixel (100,100):", reshaped_pixel_data.shape)
    
    # Compare values
    print("\nComparing first 5 years of data for pixel (100,100):")
    print("\nOriginal data (first 5 years):")
    print(original_pixel_data[:5])
    print("\nReshaped data (first 5 years):")
    print(reshaped_pixel_data[:5])
    
    # Check if data matches
    is_equal = np.allclose(original_pixel_data, reshaped_pixel_data)
    print(f"\nData match verification: {'✓ PASSED' if is_equal else '✗ FAILED'}")

verify_reshaping()

(975000000, 4)
